# SIFLOW · run_4 · MDLM ablations (fills Table 3)

Retrains short (5k-step) head variants from the same data and evaluates each at k=1,8: no-average-velocity (Di[M]O-style), hard-label (no SATD anneal), no entropy prior, +identity target, prob-space head, and head-depth. Split into two runs if a session times out (each variant resumes independently).

**Hardware:** single A100-80GB, <12h. All artifacts are written to Google Drive so the next notebook resumes.

**Needs from previous notebooks:** run_1 (tokens)

In [ ]:
# --- 1. Clone + install (run once per session) ---
REPO_URL = "https://github.com/kagtgi/siflow.git"   # <-- edit to your fork if needed
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# --- 2. Mount Drive + set artifact base (all sessions share this) ---
from siflow.utils import drive
drive.mount()
import os
os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
BASE = drive.base_dir()
print("artifacts ->", BASE)

In [ ]:
ABLATIONS = {
  "abl_no_avg_velocity": "ablation.no_avg_velocity=true",
  "abl_hard_label":      "ablation.hard_label=true",
  "abl_no_entropy_prior":"train.lam_ent=0.0",
  "abl_identity_target": "train.w_id=0.5",
  "abl_prob_space":      "head.space=prob",
  "abl_head_depth1":     "head.mid_blocks=1",
}
for rid, override in ABLATIONS.items():
    print("=== train", rid, "===")
    !python scripts/train.py --config siflow/config/mdlm.yaml --set \
        data.tokens_path={BASE}/data/owt_gpt2_256.npy out_dir={BASE}/ckpt/{rid} \
        run_id={rid} train.total_steps=5000 {override}
    !python scripts/evaluate.py --config siflow/config/mdlm.yaml --system siflow \
        --ckpt-dir {BASE}/ckpt/{rid} --ref-tokens {BASE}/data/owt_gpt2_val.npy \
        --n-samples 500 --k-list 1 8 --set run_id={rid} --out results/{rid}.json

In [ ]:
!python scripts/make_tables.py --results results
print(open('paper/tables_auto.tex').read())

In [ ]:
# --- Save outputs to Drive (so the next notebook can resume) ---
!cp -r results {BASE}/ 2>/dev/null || true
!cp -r paper/tables_auto.tex {BASE}/ 2>/dev/null || true
print('saved to', BASE)